## 4. Chia train/val/test theo bệnh nhân
- Đầu vào: `data/windows_tc/*.npz` và `data/windows_tc_summary.parquet` (từ `03_windowing.ipynb`, chỉ record 1 Hz).
- Đầu ra: `data/splits_tc.parquet` (bệnh nhân nào thuộc tập nào) và `data/dataset_tc/{train,val,test}.npz`.
- Mỗi file gồm: `X` (n, 60, **4**) với 4 kênh `spo2_mean`, `spo2_min`, `spo2_valid`, `pulse_mean` (D6; 2 kênh mask được thêm lúc train, không lưu ở đây); `Y`, `OK` (n, K) theo `T_list`; `baseline`, `baseline_hours`, `start`, `pid`, `rec`. Định dạng này khớp hợp đồng dữ liệu ở đầu `model/tc_core.py`.
- **D7:** chia theo **bệnh nhân** 70/15/15, phân tầng theo mức sự kiện của bệnh nhân: không có sự kiện ở T nào / chỉ có ở T > 88 / có ở T ≤ 88. Trong 300 seed, chọn cách chia làm tỷ lệ dương ở T = 88 và T = 92 cùng tỷ trọng cửa sổ của ba tập gần mục tiêu nhất.
- Cách chia được lưu và dùng lại. Muốn chia lại: xóa `data/splits_tc.parquet` và thư mục `data/dataset_tc`.
- Quy tắc dùng: val để chọn ngưỡng và dừng sớm; test chỉ mở **một lần** ở cuối.

In [ ]:
import os, numpy as np, pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

WIN_DIR = "data/windows_tc"; SUMMARY = f"{WIN_DIR}_summary.parquet"
SPLIT_PATH, OUT_DIR = "data/splits_tc.parquet", "data/dataset_tc"
FRAC = {"train": .70, "val": .15, "test": .15}
T_BAL = (88, 92)                                     # cân bằng tỷ lệ dương ở các T này giữa ba tập
os.makedirs(OUT_DIR, exist_ok=True)

s = pd.read_parquet(SUMMARY).query("n_win > 0")
T_LIST = np.array(sorted(int(c[3:]) for c in s.columns if c.startswith("ok_")))
pt = s.groupby("patient_id").sum(numeric_only=True)
pt["strata"] = np.select([pt[f"pos_{T_LIST.min()}"] > 0, pt["pos_88"] > 0, pt[f"pos_{T_LIST.max()}"] > 0],
                         ["co_su_kien_T_thap", "co_su_kien_T88", "chi_T_cao"], "khong_su_kien")
print(pt["strata"].value_counts().to_string())

def assign(seed):
    rng, out = np.random.default_rng(seed), pd.Series(index=pt.index, dtype=object)
    for _, idx in pt.groupby("strata").groups.items():
        idx = rng.permutation(np.array(idx)); n = len(idx)
        a, b = int(round(FRAC["train"] * n)), int(round((FRAC["train"] + FRAC["val"]) * n))
        out[idx[:a]], out[idx[a:b]], out[idx[b:]] = "train", "val", "test"
    return out

def score(sp):   # độ lệch tỷ lệ dương (ở T_BAL) và tỷ trọng cửa sổ so với mục tiêu
    g = pt.groupby(sp).sum(numeric_only=True); share = g["n_win"] / g["n_win"].sum(); dev = 0
    for T in T_BAL:
        rate, base = g[f"pos_{T}"] / g[f"ok_{T}"], pt[f"pos_{T}"].sum() / pt[f"ok_{T}"].sum()
        dev = max(dev, (rate / base - 1).abs().max())
    return dev + max(abs(share[k] - v) / v for k, v in FRAC.items())

if os.path.exists(SPLIT_PATH):
    sp = pd.read_parquet(SPLIT_PATH).set_index("patient_id")["split"]
    print(f"Dùng lại cách chia đã lưu: {SPLIT_PATH} ({len(sp)} bệnh nhân)")
else:
    best = min(range(300), key=lambda sd: score(assign(sd))); sp = assign(best)
    sp.rename("split").rename_axis("patient_id").reset_index().to_parquet(SPLIT_PATH, index=False)
    print(f"Chọn seed {best} (độ lệch {score(sp):.3f}) và lưu {SPLIT_PATH}")

s["split"] = s["patient_id"].map(sp)
for k in FRAC:
    f = f"{OUT_DIR}/{k}.npz"
    if os.path.exists(f): print(f"Bỏ qua {k} (đã có {f})"); continue
    parts = [(np.load(f"{WIN_DIR}/{r.name}.npz"), r.patient_id, r.name) for r in s[s["split"] == k].itertuples()]
    cat = lambda c: np.concatenate([z[c] for z, *_ in parts])
    out = dict(T_list=T_LIST, X=cat("X"), Y=cat("Y"), OK=cat("OK"), start=cat("start"), baseline=cat("baseline"),
               baseline_hours=cat("baseline_hours"), pid=np.concatenate([np.full(len(z["start"]), p) for z, p, _ in parts]),
               rec=np.concatenate([np.full(len(z["start"]), n) for z, _, n in parts]))
    assert all((z["T_list"] == T_LIST).all() for z, *_ in parts)
    np.savez(f"{f}.tmp.npz", **out); os.replace(f"{f}.tmp.npz", f)
    print(f"Đã lưu {k}: X {out['X'].shape}, dương ở T=88: {int((out['Y'][:, list(T_LIST).index(88)] & out['OK'][:, list(T_LIST).index(88)]).sum())}")

### 4.1 Kiểm tra
- **Assert không có bệnh nhân nào nằm ở hơn một tập**, và mỗi tập nạp được bằng `tc_core.load_split` (kiểm tra kích thước và tính lồng nhau của nhãn theo T).
- Bảng tỷ lệ dương theo T của từng tập; biểu đồ tỷ trọng mỗi tập, phân bố SpO₂ và tỷ lệ dương theo T.

In [ ]:
import sys, matplotlib.pyplot as plt
sys.path.insert(0, "../model"); import tc_core as C
DS = {k: C.load_split(OUT_DIR, k) for k in FRAC}
P = {k: set(d["pid"]) for k, d in DS.items()}
assert not (P["train"] & P["val"] or P["train"] & P["test"] or P["val"] & P["test"]), "Có bệnh nhân nằm ở nhiều tập!"
print("Không có bệnh nhân nào xuất hiện ở hơn một tập.")
LT = C.label_table(DS, T_LIST)
info = pd.DataFrame({k: dict(benh_nhan=len(P[k]), cua_so=len(d["X"]), co_baseline=np.mean(np.isfinite(d["baseline"]) & (d["baseline_hours"] >= C.BASE_MIN_HOURS)))
                     for k, d in DS.items()}).T
display(info.round(3)); display(LT.pivot(index="T", columns="split", values=["n_ok", "n_pos", "prevalence", "pat_pos"]).round(4))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2)); sp_ = list(FRAC); w = .25
for j, col in enumerate(("benh_nhan", "cua_so")):
    ax[0].bar(np.arange(3) + (j - .5) * w, [info.loc[k, col] / info[col].sum() * 100 for k in sp_], w, label=col)
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(sp_); ax[0].set_ylabel("% tổng"); ax[0].legend(); ax[0].set_title("Tỷ trọng mỗi tập (mục tiêu 70/15/15)")
for k, d in DS.items(): ax[1].hist(np.nanmean(d["X"][:, :, 0], 1), bins=np.arange(80, 100.5, .5), density=True, histtype="step", label=k)
ax[1].set(xlabel="SpO2 trung bình của cửa sổ (%)", ylabel="Mật độ", title="Phân bố SpO2 giữa các tập"); ax[1].legend()
for k, g in LT.groupby("split", sort=False): ax[2].plot(g["T"], g["prevalence"] * 100, "o-", label=k)
ax[2].set(xlabel="T (%)", ylabel="% cửa sổ dương", title="Tỷ lệ dương theo T của mỗi tập"); ax[2].legend()
plt.tight_layout(); plt.show()